In [2]:
import os
import requests
import zipfile
from pathlib import Path
import pandas as pd

In [ ]:
# ─── 1. Criar a pasta 'data' ──────────────────────────────────────────────────
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
print(f"Pasta '{data_dir}' pronta.\n")

# ─── 2. Lista de downloads ────────────────────────────────────────────────────
downloads = [
    {
        "nome": "CDC Dados",
        "url": "https://data.cdc.gov/api/views/hn4x-zwk7/rows.csv?accessType=DOWNLOAD",
        "destino": data_dir / "cdc_dados.csv",
        "zip": False,
    },
    {
        "nome": "USDA Food Prices",
        "url": "https://www.ers.usda.gov/media/5400/food-at-home-monthly-area-prices-2012-to-2018.zip?v=41906",
        "destino": data_dir / "food_prices.zip",
        "zip": True,
    },
]

# ─── 3. Função de download ────────────────────────────────────────────────────
def baixar_arquivo(nome, url, destino, headers=None):
    print(f"⬇Baixando: {nome}")
    response = requests.get(url, stream=True, headers=headers or {})
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))
    baixado = 0

    with open(destino, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            baixado += len(chunk)
            if total:
                pct = baixado / total * 100
                print(f"\r   {pct:.1f}% ({baixado:,} / {total:,} bytes)", end="")

    print(f"\nSalvo em: {destino}\n")

# ─── 4. Função de extração ZIP ────────────────────────────────────────────────
def extrair_zip(zip_path, destino_dir):
    print(f"Extraindo: {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as z:
        arquivos = z.namelist()
        print(f"   Arquivos no ZIP: {arquivos}")
        z.extractall(destino_dir)
    zip_path.unlink()
    print(f"Extraído em '{destino_dir}/' e ZIP removido.\n")

# ─── 5. Executar downloads ────────────────────────────────────────────────────
headers_gov = {"User-Agent": "Mozilla/5.0"}

for item in downloads:
    # Usa User-Agent apenas para o site .gov
    hdrs = headers_gov if "usda.gov" in item["url"] else {}
    baixar_arquivo(item["nome"], item["url"], item["destino"], headers=hdrs)

    if item["zip"]:
        extrair_zip(item["destino"], data_dir)

# ─── 6. Listar conteúdo final da pasta 'data' ─────────────────────────────────
print("Conteúdo final da pasta 'data':")
for arquivo in sorted(data_dir.iterdir()):
    tamanho = arquivo.stat().st_size
    print(f"   {arquivo.name:50s} {tamanho:>12,} bytes")

# ─── 7. Prévia dos dados ──────────────────────────────────────────────────────
print("\n─── Prévia: CDC Dados ───────────────────────────────")
df_cdc = pd.read_csv(data_dir / "cdc_dados.csv")
print(f"Linhas: {len(df_cdc):,}  |  Colunas: {len(df_cdc.columns)}")
display(df_cdc.head())

print("\n─── Prévia: USDA Food Prices ────────────────────────")
import glob
arquivos_excel = glob.glob("data/*.xlsx") + glob.glob("data/*.xls")
arquivos_csv   = [f for f in glob.glob("data/*.csv") if "cdc" not in f]

if arquivos_excel:
    df_usda = pd.read_excel(arquivos_excel[0])
elif arquivos_csv:
    df_usda = pd.read_csv(arquivos_csv[0])

print(f"Linhas: {len(df_usda):,}  |  Colunas: {len(df_usda.columns)}")
display(df_usda.head())

Pasta 'data' pronta.

⬇Baixando: CDC Dados

✅ Salvo em: data\cdc_dados.csv

⬇Baixando: USDA Food Prices
   100.0% (9,380,806 / 9,380,806 bytes)
✅ Salvo em: data\food_prices.zip

Extraindo: food_prices.zip
   Arquivos no ZIP: ['FMAP-Data.csv', 'FMAP-ReadMe.txt']
Extraído em 'data/' e ZIP removido.

Conteúdo final da pasta 'data':
   cdc_dados.csv                                        42,136,852 bytes
   FMAP-Data.csv                                        52,970,659 bytes
   FMAP-ReadMe.txt                                           9,504 bytes

─── Prévia: CDC Dados ───────────────────────────────
Linhas: 110,880  |  Colunas: 33


C:\Users\kslima\AppData\Local\Temp\ipykernel_183152\2656625346.py:76: DtypeWarning: Columns (0: Data_Value_Unit) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cdc = pd.read_csv(data_dir / "cdc_dados.csv")


,YearStart,YearEnd,LocationAbbr,LocationDesc,Datasource,Class,Topic,Question,Data_Value_Unit,Data_Value_Type,...,GeoLocation,ClassID,TopicID,QuestionID,DataValueTypeID,LocationID,StratificationCategory1,Stratification1,StratificationCategoryId1,StratificationID1
0,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$15,000 - $24,999",INC,INC1525
1,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$25,000 - $34,999",INC,INC2535
2,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$35,000 - $49,999",INC,INC3550
3,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$50,000 - $74,999",INC,INC5075
4,2011,2011,AL,Alabama,Behavioral Risk Factor Surveillance System,Obesity / Weight Status,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,NaN,Value,...,"(32.840571122, -86.631860762)",OWS,OWS1,Q036,VALUE,1,Income,"$75,000 or greater",INC,INC75PLUS



─── Prévia: USDA Food Prices ────────────────────────
Linhas: 1,020,600  |  Colunas: 6


,Year,Month,EFPG_code,Metroregion_code,Attribute,Value
0,2012,1,10000,0,Purchase_dollars_wtd,2.493734e+08
1,2012,1,10000,0,Purchase_grams_wtd,4.875195e+10
2,2012,1,10000,0,Purchase_dollars_unwtd,1.629852e+08
3,2012,1,10000,0,Purchase_grams_unwtd,3.217558e+10
4,2012,1,10000,0,Number_stores,4.178100e+04


In [3]:


# Carregar os dados
df_cdc = pd.read_csv("data/cdc_dados.csv", low_memory=False)
df_usda = pd.read_csv("data/FMAP-Data.csv")

print("=" * 60)
print("CDC — Shape:", df_cdc.shape)
print("\n--- Colunas ---")
print(df_cdc.columns.tolist())

print("\n--- Valores únicos: Class ---")
print(df_cdc["Class"].unique())

print("\n--- Valores únicos: Topic ---")
print(df_cdc["Topic"].unique())

print("\n--- Anos disponíveis ---")
print(sorted(df_cdc["YearStart"].unique()))

print("\n--- Categorias de renda ---")
print(df_cdc[df_cdc["StratificationCategory1"] == "Income"]["Stratification1"].unique())

print("\n--- Nulos por coluna (top 10) ---")
print(df_cdc.isnull().sum().sort_values(ascending=False).head(10))

CDC — Shape: (110880, 33)

--- Colunas ---
['YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc', 'Datasource', 'Class', 'Topic', 'Question', 'Data_Value_Unit', 'Data_Value_Type', 'Data_Value', 'Data_Value_Alt', 'Data_Value_Footnote_Symbol', 'Data_Value_Footnote', 'Low_Confidence_Limit', 'High_Confidence_Limit ', 'Sample_Size', 'Total', 'Age(years)', 'Education', 'Sex', 'Income', 'Race/Ethnicity', 'GeoLocation', 'ClassID', 'TopicID', 'QuestionID', 'DataValueTypeID', 'LocationID', 'StratificationCategory1', 'Stratification1', 'StratificationCategoryId1', 'StratificationID1']

--- Valores únicos: Class ---
<StringArray>
['Obesity / Weight Status', 'Physical Activity', 'Fruits and Vegetables']
Length: 3, dtype: str

--- Valores únicos: Topic ---
<StringArray>
[         'Obesity / Weight Status',     'Physical Activity - Behavior',
 'Fruits and Vegetables - Behavior']
Length: 3, dtype: str

--- Anos disponíveis ---
[np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64

In [4]:
print("=" * 60)
print("USDA — Shape:", df_usda.shape)
print("\n--- Colunas ---")
print(df_usda.columns.tolist())

print("\n--- Valores únicos: Attribute ---")
print(df_usda["Attribute"].unique())

print("\n--- Anos disponíveis ---")
print(sorted(df_usda["Year"].unique()))

print("\n--- Amostra de EFPG_code (categorias de alimento) ---")
print(df_usda["EFPG_code"].unique()[:20])

print("\n--- Amostra de Metroregion_code ---")
print(df_usda["Metroregion_code"].unique()[:20])

print("\n--- Nulos ---")
print(df_usda.isnull().sum())

USDA — Shape: (1020600, 6)

--- Colunas ---
['Year', 'Month', 'EFPG_code', 'Metroregion_code', 'Attribute', 'Value']

--- Valores únicos: Attribute ---
<StringArray>
[  'Purchase_dollars_wtd',     'Purchase_grams_wtd', 'Purchase_dollars_unwtd',
   'Purchase_grams_unwtd',          'Number_stores',    'Unit_value_mean_wtd',
      'Unit_value_se_wtd',  'Unit_value_mean_unwtd',       'Price_index_GEKS']
Length: 9, dtype: str

--- Anos disponíveis ---
[np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]

--- Amostra de EFPG_code (categorias de alimento) ---
[10000 10025 10050 10075 15000 15025 15050 15075 20000 20075 21500 21525
 21550 21575 23000 23075 24500 24525 24550 24575]

--- Amostra de Metroregion_code ---
[    0     1     2     3     4 12060 14460 16980 19100 19820 26420 31080
 33100 35620 37980]

--- Nulos ---
Year                0
Month               0
EFPG_code           0
Metroregion_code    0
Attribute           0
Val